# exp032_public_sel15_pf_residual_correction train

Train fold-held residual correction models on top of the exp029 public sel15 PF/Beam OOF-like artifact.

## Contents

1. Setup and configuration
2. Input artifact check
3. Residual correction audit
4. Metrics and artifacts


## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from residual_correction_audit import resolve_feature_path, run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

feature_path = resolve_feature_path(paths, Path(get_nested(config, "data.feature_path")))

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Feature path:", feature_path)
print("Target:", get_nested(config, "model.target"))
print("Features:", len(get_nested(config, "model.features") or []))


## 2. Input artifact check

In [ ]:
if not feature_path.exists():
    raise FileNotFoundError(f"exp029 feature artifact not found: {feature_path}")

preview = pd.read_csv(feature_path, nrows=5)
required_preview_cols = [
    "well_id",
    "fold",
    "eval_step",
    "target_tvt",
    "pf_pred",
    "last_anchor_tvt",
    "beam_pred",
    "pf_seed_std",
    "abs_pf_beam_diff",
]
print("Preview rows:", len(preview))
print("Columns:", len(preview.columns))
display(preview[required_preview_cols])


## 3. Residual correction audit

In [ ]:
summary = run_audit(paths, config, feature_path)
print(json.dumps({
    "public_pf_selector_cv": summary["public_pf_selector_cv"],
    "pf090_hold010_cv": summary["pf090_hold010_cv"],
    "best_original_fold_candidate": summary["best_original_fold_candidate"],
    "best_original_fold_cv": summary["best_original_fold_cv"],
    "best_well_hash_candidate": summary["best_well_hash_candidate"],
    "best_well_hash_cv": summary["best_well_hash_cv"],
    "selected_candidate": summary["selected_candidate"],
    "residual_model_supported": summary["residual_model_supported"],
}, indent=2))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "residual_correction_metrics.csv")
buckets = pd.read_csv(paths.artifacts_dir / "residual_correction_bucket_metrics.csv")
splits = pd.read_csv(paths.artifacts_dir / "residual_correction_split_metrics.csv")
display(metrics.head(20))
display(buckets.head(20))
display(splits.head(20))
print("Metrics written:", paths.metrics_path)
print("Artifacts written:", paths.artifacts_dir)
